# Day 29：向量数据库与 Retriever 接口（Chroma 兼容本地向量库）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 29！今天我们将实战 `LocalEmbedding`、`LocalVectorStore`（文档入库、元数据过滤、带分数检索、JSON 持久化）以及 `VectorStoreRetriever`（`as_retriever` 统一检索接口），构建企业级 RAG 系统的核心检索底座（`v0.3.0-day29`）。

## 1. 环境初始化与导入 `project_iterations/day_29` 模块
定位仓库根目录并将 `project_iterations/day_29` 加入 `sys.path`，导入 `Document`、`LocalEmbedding`、`LocalVectorStore`。

In [1]:
import sys
import tempfile
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day29_dir = repo_root / "project_iterations" / "day_29"
if str(day29_dir) not in sys.path:
    sys.path.insert(0, str(day29_dir))

from zhishu_agent import Document, LocalEmbedding, LocalVectorStore

print("已加载模块路径:", day29_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_29


## 2. 构建企业知识库并执行相似度检索（`similarity_search_with_score`）
写入带有部门、年份与密级元数据的企业文档，测试语义/词法检索排序。

In [2]:
docs = [
    Document(
        id="hr-001",
        page_content="星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。",
        metadata={"department": "hr", "year": 2025, "doc_type": "policy"},
    ),
    Document(
        id="fin-001",
        page_content="差旅报销标准：一线城市酒店住宿报销上限每晚600元，需提供电子发票。",
        metadata={"department": "finance", "year": 2025, "doc_type": "policy"},
    ),
    Document(
        id="it-001",
        page_content="生产环境 Redis 集群禁止执行 KEYS * 命令，请使用 SCAN 分批遍历。",
        metadata={"department": "it", "year": 2024, "doc_type": "runbook"},
    ),
    Document(
        id="it-002",
        page_content="智枢平台 RAG 检索默认召回 Top-3 文档片段，并支持按部门元数据过滤。",
        metadata={"department": "it", "year": 2026, "doc_type": "architecture"},
    ),
]

store = LocalVectorStore(collection_name="zhishu_kb")
ids = store.add_documents(docs)
print("入库文档 IDs:", ids)
print("集合文档总数:", store.count())

for doc, score in store.similarity_search_with_score("员工年假有多少天？", k=2):
    print(f"score={score:.4f} | id={doc.id} | {doc.page_content}")

入库文档 IDs: ['hr-001', 'fin-001', 'it-001', 'it-002']
集合文档总数: 4
score=0.2701 | id=hr-001 | 星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。
score=0.0163 | id=it-002 | 智枢平台 RAG 检索默认召回 Top-3 文档片段，并支持按部门元数据过滤。


## 3. 元数据预过滤（Metadata Filtering）与 JSON 持久化（`save / load`）
演示按 `department="it"` 与 `year >= 2025` 过滤文档，并将向量库持久化到临时 JSON 文件后重新加载。

In [3]:
filtered = store.similarity_search(
    query="检索架构",
    k=3,
    filter_metadata={"department": "it", "year": {"$gte": 2025}},
)
print("过滤后召回 ID:", [d.id for d in filtered])

tmp_dir = tempfile.TemporaryDirectory()
saved_path = store.save(Path(tmp_dir.name) / "zhishu_kb.json")
reloaded = LocalVectorStore.load(saved_path)

retriever = reloaded.as_retriever(search_kwargs={"k": 2})
ret_docs = retriever.invoke("差旅住宿报销")
print("Retriever 召回 Top-1:", ret_docs[0].id, ret_docs[0].page_content)

tmp_dir.cleanup()

过滤后召回 ID: ['it-002']
Retriever 召回 Top-1: fin-001 差旅报销标准：一线城市酒店住宿报销上限每晚600元，需提供电子发票。
